In [ ]:
!pip install --upgrade datasets fsspec transformers torch
!pip install transformers[torch]

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer
# Load IMDB dataset and subset
dataset = load_dataset("imdb")

In [ ]:
dataset

In [ ]:
train_dataset = dataset["train"].select(range(1000))
test_dataset = dataset["test"].select(range(500))

In [ ]:
# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# Tokenization function
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

In [ ]:

def preprocess(ds):
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [ ]:
train_dataset = preprocess(train_dataset)

In [ ]:
test_dataset = preprocess(test_dataset)

In [ ]:
# 3. Model load
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
for layer in model.bert.encoder.layer:
  print(layer)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./bert-finetuned-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
    push_to_hub=True,
    hub_model_id="Jim1892/IMDB-BERT-Finetuned",
    hub_strategy="end",
)

In [ ]:
# 5. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [ ]:
# 6. Train
trainer.train()

In [ ]:
tokenizer.save_pretrained("./bert-finetuned-imdb")

## Prediction

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [ ]:
# Predict
text = "This movie was amazing and I loved the acting!"
result = classifier(text)

In [ ]:
print(result)  

### Pushing it to Huggingfacehub

In [ ]:
!pip install ipywidgets

In [ ]:
from huggingface_hub import notebook_login

In [ ]:
notebook_login()

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
tokenizer.push_to_hub("Jim1892/IMDB-BERT-Finetuned")

In [ ]:
import gc

# Release old pipeline/model references that can keep a Windows mmap lock on model.safetensors.
if "classifier" in globals():
    del classifier

if "trainer" in globals() and "model" in globals() and model is not trainer.model:
    del model
    model = trainer.model

gc.collect()

In [ ]:
trainer.push_to_hub(commit_message="Upload fine-tuned IMDB BERT")